# Validation

Цель- построить локальную валидационную выборку, которая имитирует benchmark.
Для каждого валидационного поискового контекста формируется:
- набор признаков запроса
- множество релевантных `item_id`
- корпус объявлений, среди которого выполняется поиск

Качество кандидатогенерации оценивается с помощью `Recall@50`

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys

## Загрузка данных
Для построения валидации используются:
- `train.parquet` - источник поисковых запросов и известных положительных взаимодействий;
- `benchmark_items.parquet` - большой корпус реальных объявлений, который будет использоваться как источник нерелевантных кандидатов (distractors).

Использование большого корпуса позволяет сделать локальную retrieval-задачу более похожей на реальный benchmark.

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

DATA_DIR = Path("../data/raw")

train = pd.read_parquet(DATA_DIR / "train.parquet")
benchmark_items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

print("Train:", train.shape)
print("Benchmark items:", benchmark_items.shape)

Train: (497673, 19)
Benchmark items: (189212, 14)


## Определение поискового контекста
По результатам EDA один и тот же текст `search_query` может встречаться с разными локациями, категориями и фильтрами. Поэтому один поисковый случай определяется не только текстом запроса, а полной комбинацией признаков:
- `search_query`
- `search_location_id`
- `search_is_delivery_search`
- `search_infm_params_text`
- `search_category`
Для каждой уникальной комбинации создадим внутренний идентификатор `context_id`.

In [3]:
QUERY_COLS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

train = train.copy()

train["context_id"] = (
    train
    .groupby(QUERY_COLS, sort=False)
    .ngroup()
)

print(
    "Количество уникальных поисковых контекстов:",
    train["context_id"].nunique()
)

Количество уникальных поисковых контекстов: 354463


## Количество релевантных объявлений на поисковый контекст
В EDA количество релевантных объявлений анализировалось на уровне текста search_query. Для оценки retrieval-системы корректнее использовать полный поисковый контекст. Поэтому проверим, сколько уникальных положительных `item_id` приходится на каждый context_id

In [4]:
relevant_per_context = (
    train
    .groupby("context_id")["item_id"]
    .nunique()
)

display(
    relevant_per_context.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

print(
    "Контекстов с 1 релевантным объявлением:",
    f"{(relevant_per_context == 1).mean():.2%}"
)

print(
    "Контекстов с <= 2 релевантными объявлениями:",
    f"{(relevant_per_context <= 2).mean():.2%}"
)

print(
    "Контекстов с <= 5 релевантными объявлениями:",
    f"{(relevant_per_context <= 5).mean():.2%}"
)

count    354463.000000
mean          1.317638
std           1.611180
min           1.000000
50%           1.000000
75%           1.000000
90%           2.000000
95%           3.000000
99%           6.000000
max         278.000000
Name: item_id, dtype: float64

Контекстов с 1 релевантным объявлением: 85.00%
Контекстов с <= 2 релевантными объявлениями: 94.49%
Контекстов с <= 5 релевантными объявлениями: 98.80%


## Стратегия разделения train и val

Обычный случайный `train_test_split` по строкам использовать тут нежелательно. Один и тот же текст запроса может встречаться в train сотни или тысячи раз. При случайном разделении взаимодействия одного запроса окажутся одновременно в обучающей и валидационной частях, что может привести к завышенной оценке качества retrieval-системы.
Для основной локальной валидации используется более строгий сценарий unseen queries:
- случайно выбирается набор уникальных текстов `search_query`
- все взаимодействия этих запросов полностью переносятся в validation
- эти тексты не встречаются в retrieval train

Таким образом проверяется способность системы находить релевантные объявления для ранее не встречавшихся формулировок запросов. Benchmark содержит как ранее встречавшиеся, так и новые тексты запросов, поэтому в дальнейшем отдельно можно добавить validation для seen queries.

In [5]:
RANDOM_STATE = 42
VALID_SIZE = 2452

unique_queries = train["search_query"].drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)

valid_query_texts = rng.choice(
    unique_queries,
    size=VALID_SIZE,
    replace=False
)

valid_query_texts = set(valid_query_texts)

print("Выбрано текстов запросов:", len(valid_query_texts))

Выбрано текстов запросов: 2452


## Формирование train и validation

Все взаимодействия выбранных search_query переносятся в validation. Остальные взаимодействия остаются в `train_retrieval` и в дальнейшем могут использоваться для методов, основанных на истории взаимодействий. После разделения дополнительно проверяется отсутствие пересечения текстов запросов между двумя частями.

In [6]:
is_valid = train["search_query"].isin(valid_query_texts)

train_retrieval = train.loc[~is_valid].copy()
valid_interactions = train.loc[is_valid].copy()

print("Retrieval train:", train_retrieval.shape)
print("Validation interactions:", valid_interactions.shape)

Retrieval train: (483038, 20)
Validation interactions: (14635, 20)


In [7]:
query_overlap = (
    set(train_retrieval["search_query"])
    & set(valid_interactions["search_query"])
)

print(
    "Пересечение текстов запросов между train и validation:",
    len(query_overlap)
)

assert len(query_overlap) == 0

Пересечение текстов запросов между train и validation: 0


## Формирование validation queries
Из положительных взаимодействий оставим только признаки поискового запроса. Каждая уникальная комбинация признаков соответствует одному context_id и играет роль аналога query_id из настоящего benchmark.

In [8]:
validation_queries = (
    valid_interactions[
        ["context_id"] + QUERY_COLS
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Количество validation contexts:",
    len(validation_queries)
)

display(validation_queries.head())

Количество validation contexts: 11476


,context_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,37,строительная бригада,657630,0,Вид услуги Строительство,114
1,78,трансфер краснодар,621585,0,Вид услуги Пассажирские перевозки,114
2,92,гидрообразивная резка керамогранита,653241,0,,114
3,116,перманент губ,660630,0,"Вид услуги Красота, здоровье",114
4,126,hydrafacial,637640,0,"Онлайн-запись Вид услуги Красота, здоровье",114


## Формирование ground truth
Для каждого `context_id` сохраним множество объявлений, которые пользователи действительно выбирали в train. Эти данные представляют собой локальный аналог скрытой benchmark-разметки. Retrieval-модель не использует `validation_labels` при поиске кандидатов — они необходимы только для последующего вычисления Recall@50

In [9]:
validation_labels = (
    valid_interactions
    .groupby("context_id")["item_id"]
    .agg(lambda x: list(dict.fromkeys(x)))
    .to_dict()
)

print(
    "Количество контекстов с ground truth:",
    len(validation_labels)
)

Количество контекстов с ground truth: 11476


In [10]:
for context_id in list(validation_labels.keys())[:5]:
    print(
        f"{context_id}:",
        validation_labels[context_id]
    )

37: ['27ad9f013a895bdf']
78: ['19f7044cada5f537']
92: ['91af2b38b968fb23']
116: ['dfb97045d616f5bd']
126: ['391d0fd6b4b6281b']


In [11]:
assert len(validation_labels) == len(validation_queries)

## Формирование корпуса объявлений

В реальной задаче поиск выполняется среди 189 212 объявлений из `benchmark_items.parquet`. Использовать только объявления из validation-взаимодействий нельзя: такой корпус содержал бы преимущественно положительные примеры и сделал бы retrieval-задачу нереалистично простой.

Поэтому в качестве основы используется настоящий `benchmark_items` corpus. К нему добавляются положительные объявления validation, которых может не быть среди benchmark items. После объединения удаляются дубликаты по item_id`.

In [12]:
ITEM_COLS = [
    "item_title_raw",
    "item_rating_reviews_count",
    "item_rating",
    "item_price",
    "item_microcat_id",
    "item_longitude",
    "item_location_id",
    "item_latitude",
    "item_is_phone_hidden",
    "item_is_message_forbidden",
    "item_infm_params_text",
    "item_id",
    "item_description_raw",
    "item_category_id",
]

In [13]:
validation_positive_items = (
    valid_interactions[ITEM_COLS]
    .drop_duplicates(subset="item_id")
)

print(
    "Уникальных positive validation items:",
    len(validation_positive_items)
)

Уникальных positive validation items: 13270


In [14]:
validation_items = pd.concat(
    [
        benchmark_items,
        validation_positive_items
    ],
    ignore_index=True
)

validation_items = (
    validation_items
    .drop_duplicates(subset="item_id")
    .reset_index(drop=True)
)

print(
    "Размер validation corpus:",
    len(validation_items)
)

Размер validation corpus: 201651


## Проверка покрытия ground truth
Для корректного вычисления Recall каждый релевантный `item_id` должен присутствовать в корпусе, среди которого выполняется поиск. Проверяем что все объявления из validation_labels существуют в validation_items

In [15]:
corpus_ids = set(validation_items["item_id"])

all_relevant_ids = {
    item_id
    for relevant_items in validation_labels.values()
    for item_id in relevant_items
}

missing_items = all_relevant_ids - corpus_ids

print("Уникальных relevant items:", len(all_relevant_ids))
print("Отсутствуют в validation corpus:", len(missing_items))

assert len(missing_items) == 0

Уникальных relevant items: 13270
Отсутствуют в validation corpus: 0


## Метрика Recall@50

Целевая метрика задачи - `Recall@50`. Для каждого поискового контекста вычисляется доля релевантных объявлений, которые попали в первые 50 кандидатов. Затем Recall усредняется по всем запросам. Перед использованием метрики на реальных predictions проверим реализацию на простом примере.

In [16]:
PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.metrics import recall_at_k, mean_recall_at_k

In [17]:
# проверяем
assert recall_at_k(
    predictions=["a", "b", "c"],
    relevant_items=["b", "d"],
    k=3,
) == 0.5

assert recall_at_k(
    predictions=["a", "b"],
    relevant_items=["a"],
    k=50,
) == 1.0

assert recall_at_k(
    predictions=["a", "b"],
    relevant_items=["c"],
    k=50,
) == 0.0

print("Все тесты Recall@K пройдены.")

Все тесты Recall@K пройдены.


## Итоги

В результате построена локальная валидационная схема для оценки методов кандидатогенерации.

- Исходная обучающая выборка содержит 354 463 уникальных поисковых контекста
- 85% поисковых контекстов имеют только одно наблюдаемое релевантное объявление, 94,5% - не более двух, а 98,8% - не более пяти
- Для validation было случайно выбрано 2 452 уникальных текста поисковых запросов
- Все взаимодействия выбранных запросов были полностью исключены из retrieval train, поэтому пересечение текстов запросов между train и validation отсутствует
- Выбранные тексты соответствуют 11 476 уникальным поисковым контекстам
- Ground truth содержит 13 270 уникальных релевантных объявлений
- Validation corpus содержит 201 651 объявление и по масштабу сопоставим с benchmark-корпусом из 189 212 объявлений
- Все релевантные validation-объявления присутствуют в корпусе, поэтому Recall@50 может быть корректно рассчитан для каждого поискового контекста.
- Реализация Recall@50 дополнительно проверена на простых тестовых примерах

Используемая схема представляет собой строгую unseen-query validation: тексты validation-запросов полностью отсутствуют в retrieval train. Поэтому она в первую очередь оценивает способность retrieval-модели обобщаться на новые поисковые запросы.

При этом реальный benchmark содержит как новые, так и ранее встречавшиеся в train запросы. Поэтому полученный offline Recall не следует интерпретировать как точную оценку leaderboard Recall. В дальнейшем качество historical retrieval целесообразно дополнительно оценивать на отдельной seen-query validation.